# Step 13 — ADP-A Dataset Preparation
**Phase:** 4 — Model Training & Adapter Alignment  
**Spec authority:** SPEC-400 §3.2, §7.0.1, §7.0.3; REQ-400-LF2, REQ-400-CL1, REQ-400-171  
**Prerequisites:** Step 12 complete (`finetuning/adp_c_evaluator/adp_c_final/` exists and smoke-tested)  
**Output:** `finetuning/adp_a_empathy/data/adp_a_train.jsonl`

---

## Purpose

This notebook constructs the supervised fine-tuning corpus for **ADP-A**, the Empathy Layer adapter. ADP-A's training objective, as defined in SPEC-400 §3.2, is to teach the base model emotional resonance, active listening, and validation language — without crossing into clinical authority or diagnostic labelling. The output of this notebook (`adp_a_train.jsonl`) is the direct input to Step 14 (ADP-A QLoRA Training).

## Dataset Mix

ADP-A is trained on a weighted blend of five open-licence corpora, ratified in REQ-400-LF2 (2026-05-10):

| Dataset | Weight | Tier | Rationale |
|---------|--------|------|-----------|
| AnnoMI | 30% | 1 | Only publicly available expert-annotated MI corpus. Highest signal quality in the stack. |
| Amod / mental_health_counseling_conversations | 25% | 1 | Real anonymised Q&A from licensed counselors. RAIL-D licence — non-commercial research confirmed. |
| ESConv | 20% | 2 | ACL 2021 dataset; strategy-labelled utterances teach support sequencing. CC-BY-NC-4.0. |
| MentalChat16K | 15% | 3 | Synthetic; broadest topic coverage (33 topics). Must pass ADP-C filter (REQ-400-171). MIT licence. |
| EmpatheticDialogues | 10% | 4 | Short empathetic acknowledgments. Tonal calibration at tail of mix. CC-BY-NC-4.0. |

Full licence details and provenance are in `finetuning/dataset_registry.yaml`.

## ADP-C Rejection Sampling

Every candidate (instruction, response) pair is scored by the ADP-C evaluator before it enters the training set. Only pairs scoring at or above `ACCEPT_THRESHOLD` are kept. This ensures ADP-A trains on the *good* distribution — responses that already pass the system's own quality oracle — rather than the raw noisy output of the corpora.

`ACCEPT_THRESHOLD` is set to **0.50** (lowered from 0.70 after G-DATA-06, Director-ratified 2026-05-11). ADP-C was calibrated on synthetic redline pairs and systematically under-scores organic empathy exchanges at 0.70. 0.50 retains meaningful filtering while not penalising natural conversational register.

**AnnoMI exemption:** AnnoMI (Tier 1, expert-annotated) is exempted from ADP-C filtering. Pass rate was ~6% at both 0.50 and 0.70 thresholds — a confirmed calibration mismatch, not a quality failure. AnnoMI records are top-filled directly to target after the main filter pass (see Section 5).

## Hardware Note (RTX 3070 — 8 GB VRAM)

The ADP-C model is loaded in 4-bit NF4 quantization for the filter pass. `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` is set in Cell 0 before any CUDA operation, and the 5000 MiB GPU cap leaves headroom for the quantization load spike. The filter pass takes approximately 20–25 minutes on an RTX 3070 for a candidate pool of ~2000 pairs.

## Section 1 — Environment Setup

In [1]:
import os
# [CONCEPT] PYTORCH_CUDA_ALLOC_CONF — tells PyTorch to use expandable memory
# segments instead of fixed-size blocks. Prevents OOM on Windows/WDDM where
# the GPU driver can't reclaim fragmented allocations between ops.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"]   = "false"

# [CONCEPT] Import order on Windows: datasets and trl MUST load before torch.
# Both libraries register pyarrow workers at import time. If torch initialises
# CUDA first, pyarrow's multiprocessing fork fails with a CUDA re-init error.
import datasets
import trl
import torch
print(f"CUDA available : {torch.cuda.is_available()}")
print(f"Device         : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


C:\Users\dalet\anaconda3\envs\nikko\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA available : True
Device         : NVIDIA GeForce RTX 3070


In [2]:
import json, random, re, pathlib
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)


In [3]:
BASE_DIR    = pathlib.Path(r"D:/Git Repos/nikko-companion")
FINETUNING  = BASE_DIR / "finetuning"
ADP_C_DIR   = FINETUNING / "adp_c_evaluator" / "adp_c_final"
OUT_DIR     = FINETUNING / "adp_a_empathy" / "data"
OUT_FILE    = OUT_DIR / "adp_a_train.jsonl"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"ADP-C adapter : {ADP_C_DIR}")
print(f"Output file   : {OUT_FILE}")


ADP-C adapter : D:\Git Repos\nikko-companion\finetuning\adp_c_evaluator\adp_c_final
Output file   : D:\Git Repos\nikko-companion\finetuning\adp_a_empathy\data\adp_a_train.jsonl


## Section 2 — Load ADP-C Evaluator

ADP-C is loaded here to act as the rejection sampling oracle. It scores each candidate (instruction, response) pair and accepts or rejects it before it enters the training set. The base model is loaded in 4-bit NF4 quantization to fit within the RTX 3070 VRAM budget; only the ADP-C adapter weights are kept in full precision on top.

In [4]:
BASE_MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading base model (4-bit)...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    max_memory={0: "5000MiB", "cpu": "16GiB"},
    torch_dtype=torch.bfloat16,
)

print("Loading ADP-C adapter...")
adp_c_model = PeftModel.from_pretrained(base_model, str(ADP_C_DIR))
adp_c_model.eval()
print("ADP-C ready.")


Loading tokenizer...
Loading base model (4-bit)...


C:\Users\dalet\anaconda3\envs\nikko\Lib\site-packages\accelerate\utils\modeling.py:329: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\cb\pytorch_1000000000000\work\c10/cuda/CUDAAllocatorConfig.h:28.)
  new_value = value.to(device)
Loading checkpoint shards: 100%|█████████████████| 3/3 [00:17<00:00,  5.73s/it]


Loading ADP-C adapter...
ADP-C ready.


In [5]:
ACCEPT_THRESHOLD = 0.5  # Director-ratified 2026-05-11 (G-DATA-06).
                                 # Lowered from 0.70 — ADP-C calibrated on synthetic redline
                                 # pairs; organic empathy datasets scored 1-12% at 0.70.

def adp_c_score(instruction: str, response: str) -> float:
    """
    Score a (instruction, response) pair on empathy quality.
    Returns float in [0, 1]. Accepts if >= ACCEPT_THRESHOLD.
    """
    prompt = (
        f"[INST] Rate the empathy quality of this response on a scale of 0 to 1.\n"
        f"Instruction: {instruction}\n"
        f"Response: {response}\n"
        f"Reply with only a decimal number between 0 and 1. [/INST]"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to("cuda")
    with torch.no_grad():
        output = adp_c_model.generate(
            **inputs,
            max_new_tokens=8,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    match = re.search(r"[0-9]+\.?[0-9]*", generated)
    if match:
        return min(max(float(match.group()), 0.0), 1.0)
    return 0.0


## Section 3 — Dataset Loaders

Each loader extracts (instruction, output) pairs from its HuggingFace source up to `candidate_cap`. All five datasets have schema quirks discovered during Phase 4 Step 13 development — fixes are documented inline.

**Dataset mix config** is defined first, followed by one loader cell per dataset.

In [6]:
# Dataset mix weights from config.yaml / REQ-400-LF2.
# candidate_cap: max pairs to extract before ADP-C filtering.
# target: desired records in final JSONL after filtering + top-up.
DATASET_MIX = [
    {"id": "annomi",     "weight": 0.30, "candidate_cap": 600, "target": 240,
     "hf_name": "to-be/annomi-motivational-interviewing-therapy-conversations"},
    {"id": "amod",       "weight": 0.25, "candidate_cap": 500, "target": 200,
     "hf_name": "Amod/mental_health_counseling_conversations"},
    {"id": "esconv",     "weight": 0.20, "candidate_cap": 400, "target": 160,
     "hf_name": "thu-coai/esconv"},
    {"id": "mentalchat", "weight": 0.15, "candidate_cap": 300, "target": 120,
     "hf_name": "ShenLab/MentalChat16K"},
    {"id": "empathetic", "weight": 0.10, "candidate_cap": 200, "target": 80,
     "hf_name": "facebook/empathetic_dialogues"},
]
print("Dataset mix loaded.")


Dataset mix loaded.


In [7]:
# AnnoMI — expert-annotated motivational interviewing transcripts.
# Schema fix: HF uses ShareGPT-style {'from': role, 'value': text} per turn.
# Short-form 'AnnoMI' returns 404 — must use full dataset ID (G-DATA-03).

CLIENT_ROLES    = {"client", "human", "user", "usr", "patient"}
THERAPIST_ROLES = {"therapist", "assistant", "gpt", "counselor", "counsellor", "sys"}

def load_annomi(cfg):
    ds = load_dataset(cfg["hf_name"], split="train", trust_remote_code=True)
    pairs = []
    for row in ds:
        convs = row.get("conversations") or []
        for i in range(len(convs) - 1):
            prev, curr = convs[i], convs[i + 1]
            if (prev.get("from") or "").lower().strip() not in CLIENT_ROLES:
                continue
            if (curr.get("from") or "").lower().strip() not in THERAPIST_ROLES:
                continue
            user_msg  = (prev.get("value") or prev.get("text") or "").strip()
            therapist = (curr.get("value") or curr.get("text") or "").strip()
            if not user_msg or not therapist:
                continue
            pairs.append({"instruction": user_msg, "output": therapist})
            if len(pairs) >= cfg["candidate_cap"]:
                break
        if len(pairs) >= cfg["candidate_cap"]:
            break
    random.shuffle(pairs)
    print(f"[annomi] {len(pairs)} candidate pairs extracted (cap={cfg['candidate_cap']})")
    return pairs

annomi_candidates = load_annomi(DATASET_MIX[0])


[annomi] 600 candidate pairs extracted (cap=600)


In [8]:
# Amod/mental_health_counseling_conversations
# Schema fix: LICENSE-RAIL-D.txt causes DatasetGenerationCastError when the
# builder picks it up alongside data files. Fix: data_files glob to *.json only.

def load_amod(cfg):
    ds = None
    for kwargs in [
        {"data_files": {"train": "*.json"}, "split": "train"},
        {"data_files": {"train": "combined_dataset.json"}, "split": "train"},
        {"data_files": {"train": "data.json"}, "split": "train"},
        {"data_files": {"train": "train.json"}, "split": "train"},
    ]:
        try:
            ds = load_dataset(cfg["hf_name"], trust_remote_code=True, **kwargs)
            print(f"[amod] Loaded via {kwargs}")
            break
        except Exception as e:
            print(f"[amod] Strategy failed: {e}")
    if ds is None:
        print("[amod] All strategies failed — skipping")
        return []
    pairs = []
    for row in ds:
        instruction = (row.get("Context") or row.get("context") or row.get("question") or "").strip()
        output      = (row.get("Response") or row.get("response") or row.get("answer") or "").strip()
        if not instruction or not output:
            continue
        pairs.append({"instruction": instruction, "output": output})
        if len(pairs) >= cfg["candidate_cap"]:
            break
    random.shuffle(pairs)
    print(f"[amod] {len(pairs)} candidate pairs extracted (cap={cfg['candidate_cap']})")
    return pairs

amod_candidates = load_amod(DATASET_MIX[1])


[amod] Loaded via {'data_files': {'train': '*.json'}, 'split': 'train'}
[amod] 500 candidate pairs extracted (cap=500)


In [9]:
# ESConv (Emotional Support Conversation)
# Schema fix: HF version wraps each conversation as a JSON string in a 'text'
# column rather than exposing a flat dict. Parse text -> dict -> dialog list.

def load_esconv(cfg):
    ds = load_dataset(cfg["hf_name"], split="train", trust_remote_code=True)
    pairs = []
    for conv in ds:
        raw = conv.get("dialog") or conv.get("text") or ""
        if isinstance(raw, str):
            try:
                parsed = json.loads(raw)
            except json.JSONDecodeError:
                continue
            dialog = parsed.get("dialog", []) if isinstance(parsed, dict) else (parsed if isinstance(parsed, list) else [])
        elif isinstance(raw, list):
            dialog = raw
        else:
            continue
        for i in range(len(dialog) - 1):
            turn_a, turn_b = dialog[i], dialog[i + 1]
            spk_a = (turn_a.get("speaker") or turn_a.get("role") or "").lower()
            spk_b = (turn_b.get("speaker") or turn_b.get("role") or "").lower()
            if spk_a not in ("usr", "user", "human") or spk_b not in ("sys", "system", "assistant"):
                continue
            user_msg = (turn_a.get("text") or turn_a.get("content") or "").strip()
            support  = (turn_b.get("text") or turn_b.get("content") or "").strip()
            if not user_msg or not support:
                continue
            pairs.append({"instruction": user_msg, "output": support})
            if len(pairs) >= cfg["candidate_cap"]:
                break
        if len(pairs) >= cfg["candidate_cap"]:
            break
    random.shuffle(pairs)
    print(f"[esconv] {len(pairs)} candidate pairs extracted (cap={cfg['candidate_cap']})")
    return pairs

esconv_candidates = load_esconv(DATASET_MIX[2])


[esconv] 400 candidate pairs extracted (cap=400)


In [10]:
# MentalChat16K — synthetic multi-turn counselor/client conversations.
# Cleanest schema in the stack. REQ-400-171: must pass ADP-C filter.

def load_mentalchat(cfg):
    ds = load_dataset(cfg["hf_name"], split="train", trust_remote_code=True)
    pairs = []
    for row in ds:
        instruction = (row.get("input") or row.get("instruction") or row.get("question") or "").strip()
        output      = (row.get("output") or row.get("response") or row.get("answer") or "").strip()
        if not instruction or not output:
            continue
        pairs.append({"instruction": instruction, "output": output})
        if len(pairs) >= cfg["candidate_cap"]:
            break
    random.shuffle(pairs)
    print(f"[mentalchat] {len(pairs)} candidate pairs extracted (cap={cfg['candidate_cap']})")
    return pairs

mentalchat_candidates = load_mentalchat(DATASET_MIX[3])


[mentalchat] 300 candidate pairs extracted (cap=300)


In [11]:
# EmpatheticDialogues (Facebook AI)
# Schema fix: utterance_idx is non-zero in current HF version — index filter
# skips all rows. Replaced with conv_id deduplication (first row per conversation).

def load_empathetic(cfg):
    ds = load_dataset(cfg["hf_name"], split="train", trust_remote_code=True)
    seen_convs = set()
    pairs = []
    for row in ds:
        conv_id = row.get("conv_id", "")
        if conv_id in seen_convs:
            continue
        seen_convs.add(conv_id)
        user_msg = (row.get("prompt") or "").strip().replace("_comma_", ",")
        response = (row.get("utterance") or "").strip().replace("_comma_", ",")
        if not user_msg or not response:
            continue
        pairs.append({"instruction": user_msg, "output": response})
        if len(pairs) >= cfg["candidate_cap"]:
            break
    random.shuffle(pairs)
    print(f"[empathetic] {len(pairs)} candidate pairs extracted (cap={cfg['candidate_cap']})")
    return pairs

empathetic_candidates = load_empathetic(DATASET_MIX[4])


[empathetic] 200 candidate pairs extracted (cap=200)


## Section 4 — ADP-C Rejection Sampling Filter

Each candidate pool is passed through `adp_c_score()`. Pairs scoring below `ACCEPT_THRESHOLD` (0.50) are discarded. Progress is logged every 50 examples.

AnnoMI is included here for pass-rate logging only. Its filtered results are **not used** in final assembly — the top-up cell in Section 5 fills AnnoMI directly to target without ADP-C screening (G-DATA-06 exemption).

In [12]:
all_candidates = [
    (DATASET_MIX[0], annomi_candidates),
    (DATASET_MIX[1], amod_candidates),
    (DATASET_MIX[2], esconv_candidates),
    (DATASET_MIX[3], mentalchat_candidates),
    (DATASET_MIX[4], empathetic_candidates),
]

filtered = {}

for cfg, candidates in all_candidates:
    ds_id    = cfg["id"]
    accepted = []
    for i, pair in enumerate(candidates):
        score = adp_c_score(pair["instruction"], pair["output"])
        if score >= ACCEPT_THRESHOLD:
            accepted.append(pair)
        if (i + 1) % 50 == 0:
            pct = 100 * len(accepted) / (i + 1)
            print(f"  [{ds_id}] {i+1}/{len(candidates)}  pass={len(accepted)} ({pct:.0f}%)...")
    pass_pct = 100 * len(accepted) / len(candidates) if candidates else 0
    print(f"  [{ds_id}] Done. Pass: {len(accepted)} ({pass_pct:.1f}%)  Reject: {len(candidates)-len(accepted)}")
    filtered[ds_id] = accepted


  [annomi] 50/600  pass=2 (4%)...
  [annomi] 100/600  pass=3 (3%)...
  [annomi] 150/600  pass=8 (5%)...
  [annomi] 200/600  pass=11 (6%)...
  [annomi] 250/600  pass=15 (6%)...
  [annomi] 300/600  pass=18 (6%)...
  [annomi] 350/600  pass=20 (6%)...
  [annomi] 400/600  pass=21 (5%)...
  [annomi] 450/600  pass=28 (6%)...
  [annomi] 500/600  pass=31 (6%)...
  [annomi] 550/600  pass=37 (7%)...
  [annomi] 600/600  pass=37 (6%)...
  [annomi] Done. Pass: 37 (6.2%)  Reject: 563
  [amod] 50/500  pass=26 (52%)...
  [amod] 100/500  pass=38 (38%)...
  [amod] 150/500  pass=54 (36%)...
  [amod] 200/500  pass=77 (38%)...
  [amod] 250/500  pass=96 (38%)...
  [amod] 300/500  pass=115 (38%)...
  [amod] 350/500  pass=135 (39%)...
  [amod] 400/500  pass=157 (39%)...
  [amod] 450/500  pass=175 (39%)...
  [amod] 500/500  pass=198 (40%)...
  [amod] Done. Pass: 198 (39.6%)  Reject: 302
  [esconv] 50/400  pass=7 (14%)...
  [esconv] 100/400  pass=17 (17%)...
  [esconv] 150/400  pass=29 (19%)...
  [esconv] 200/40

## Section 5 — Assembly & AnnoMI Top-up

The filtered pools are sampled down to each dataset's target count and written to `adp_a_train.jsonl`. AnnoMI is then top-filled directly from the full candidate pool (no ADP-C filter) to reach its 240-record target, per the G-DATA-06 exemption.

In [13]:
rows = []
sampled_all = []

# ── Post-assembly quality filters ─────────────────────────────────────────────
# Applied after ADP-C filtering to catch artifacts that passed the scorer.

import unicodedata

TURN_MARKERS = re.compile(
    r"(^|\s)(User:|Nikko:|Human:|Assistant:|Client:|Therapist:)|︙|"
    r"\[Operating Mode|<s>|\[INST\]|\[/INST\]",
    re.IGNORECASE,
)

def is_english(text: str) -> bool:
    """
    Rejects records where >15% of characters are non-Latin/non-ASCII.
    Catches Swedish, Danish, and other Latin-adjacent artifacts while
    allowing common punctuation and Unicode quotes.
    """
    if not text:
        return False
    non_latin = sum(
        1 for c in text
        if unicodedata.category(c).startswith("L")
        and ord(c) > 591   # Latin Extended-B upper bound; anything above is non-Latin
    )
    return (non_latin / max(len(text), 1)) < 0.15

def is_clean(record: dict) -> bool:
    instruction = record.get("instruction", "")
    output      = record.get("output", "")
    # Reject multi-turn markers in either field
    if TURN_MARKERS.search(output) or TURN_MARKERS.search(instruction):
        return False
    # Reject non-English
    if not is_english(output) or not is_english(instruction):
        return False
    # Reject outputs too short to carry empathy signal (< 8 words)
    if len(output.split()) < 8:
        return False
    return True

for cfg in DATASET_MIX:
    ds_id     = cfg["id"]
    target    = cfg["target"]
    available = filtered.get(ds_id, [])
    # Apply quality filter before sampling
    clean     = [r for r in available if is_clean(r)]
    rejected  = len(available) - len(clean)
    sampled   = clean[:target]
    short     = target - len(sampled)
    status    = f"SHORT by {short}" if short > 0 else "OK"
    rows.append([ds_id, target, len(clean), len(sampled), status, rejected])
    # Tag each record with its source dataset before extending.
    # Required by Step 14's distribution audit (SPEC-400 §7.0.1).
    for r in sampled:
        r["source"] = ds_id
    sampled_all.extend(sampled)

random.shuffle(sampled_all)

total_target = sum(c["target"] for c in DATASET_MIX)
print(f"Assembling final dataset (target: {total_target})...")
print(f"{'Dataset':<22} {'Target':>8} {'Clean':>8} {'Sampled':>10} {'Dropped':>9}  Status")
print("-" * 72)
for r in rows:
    print(f"{r[0]:<22} {r[1]:>8} {r[2]:>8} {r[3]:>10} {r[4+1]:>9}  {r[4]}")
print("-" * 72)
print(f"{'TOTAL':<22} {'':>8} {'':>8} {len(sampled_all):>10}  (pre-topup)")

with open(OUT_FILE, "w", encoding="utf-8") as f:
    for rec in sampled_all:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

missing_i = sum(1 for r in sampled_all if not r.get("instruction"))
missing_o = sum(1 for r in sampled_all if not r.get("output"))
print(f"\nRecords written (pre-topup): {len(sampled_all)}")
print(f"Missing instruction: {missing_i}  |  Missing output: {missing_o}")


Assembling final dataset (target: 800)...
Dataset                  Target    Clean    Sampled   Dropped  Status
------------------------------------------------------------------------
annomi                      240       36         36         1  SHORT by 204
amod                        200      198        198         0  SHORT by 2
esconv                      160       67         67         6  SHORT by 93
mentalchat                  120      211        120         0  OK
empathetic                   80       21         21         1  SHORT by 59
------------------------------------------------------------------------
TOTAL                                           442  (pre-topup)

Records written (pre-topup): 442
Missing instruction: 0  |  Missing output: 0


In [14]:
# AnnoMI top-up — G-DATA-06 exemption, Director-ratified 2026-05-11.
# ADP-C pass rate for AnnoMI was ~6% at both 0.50 and 0.70 thresholds.
# Root cause: calibration mismatch (MI dialogue style vs synthetic training data).
# Tier 1 expert-annotated source is filled directly to target.

ANNOMI_TARGET  = 240
# Use the actual clean sampled count from the assembly pass, not the raw filtered count.
annomi_sampled = next(r[3] for r in rows if r[0] == "annomi")
topup_needed   = max(0, ANNOMI_TARGET - annomi_sampled)
print(f"AnnoMI in filtered set : {annomi_sampled}")
print(f"Top-up needed          : {topup_needed}")

existing_lines = [l for l in OUT_FILE.read_text(encoding="utf-8").splitlines() if l.strip()]
assert len(existing_lines) == len(sampled_all), (
    f"File contains {len(existing_lines)} records but sampled_all has {len(sampled_all)}. "
    "Re-run cell_assemble_13 before re-running this cell to avoid duplicate records."
)

if topup_needed > 0:
    already_texts = {r["instruction"] for r in filtered.get("annomi", [])}
    topup_pool = [r for r in annomi_candidates if r["instruction"] not in already_texts and is_clean(r)]
    random.shuffle(topup_pool)
    topup_records = topup_pool[:topup_needed]
    with open(OUT_FILE, "a", encoding="utf-8") as f:
        for rec in topup_records:
            rec["source"] = "annomi"   # tag topup records identically to assembly pass
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print(f"Appended {len(topup_records)} AnnoMI records (no ADP-C filter).")
else:
    print("AnnoMI already at target — no top-up needed.")

# Final validation
final_records = [json.loads(l) for l in OUT_FILE.read_text(encoding="utf-8").splitlines()]
file_size_kb  = OUT_FILE.stat().st_size / 1024
missing_i     = sum(1 for r in final_records if not r.get("instruction"))
missing_o     = sum(1 for r in final_records if not r.get("output"))

print(f"\nFinal total records : {len(final_records)}")
print(f"File size           : {file_size_kb:.1f} KB")
print(f"Missing instruction : {missing_i}  |  Missing output: {missing_o}")
if missing_i == 0 and missing_o == 0:
    print("\nAll records valid. adp_a_train.jsonl is ready for Step 14.")
else:
    print("\nWARNING: missing fields detected. Investigate before Step 14.")


AnnoMI in filtered set : 36
Top-up needed          : 204
Appended 204 AnnoMI records (no ADP-C filter).

Final total records : 646
File size           : 506.3 KB
Missing instruction : 0  |  Missing output: 0

All records valid. adp_a_train.jsonl is ready for Step 14.
